# House price pipeline — demo

Walks the pipeline end to end on the synthetic extract.

Generate the data first (from the repo root):

```
uv run python house_price_prediction/scripts/generate_synthetic.py \
    --out house_price_prediction/datasets/houses_synthetic.csv
```


In [1]:
from pathlib import Path

import pandas as pd

from house_price_prediction import TARGET, Config, __version__
from house_price_prediction.artifacts import write_artifacts
from house_price_prediction.benchmark import benchmark
from house_price_prediction.data import load_data, prepare, train_filter, validate
from house_price_prediction.model import base_params, cv_score, fit_final, tune
from house_price_prediction.splits import make_splits

print("package version:", __version__)
DATA = Path("../datasets/houses_synthetic.csv")

package version: 0.1.0


/Users/keerthiningegowda/Desktop/debugging_mode_for_DS/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load and validate

`validate` raises on anything that would corrupt the model or make MAPE meaningless (nulls, non-positive prices, negative counts, a proportion outside [0, 1]).

In [2]:
cfg = Config()
df = load_data(str(DATA), cfg)
validate(df, cfg)
df.head()

[validate] ok: 3000 rows, 2.7% mean missingness


,house_price,dwelling_size,property_age,dwelling_style,bedrooms,bathrooms,prop_visible_minorities
0,715951.699009,1982.830248,63.0,detached,1.0,1.0,0.472892
1,503669.881261,1176.009536,115.0,condo,1.0,3.0,0.192142
2,473221.860370,2250.270717,8.0,condo,3.0,2.0,0.112416
3,887261.325748,2364.338830,97.0,detached,1.0,1.0,0.472448
4,292509.995584,629.378887,64.0,townhouse,1.0,1.0,0.127813


In [3]:
df[cfg.features].isna().mean().sort_values(ascending=False)

property_age               0.078667
dwelling_size              0.052667
bathrooms                  0.030000
dwelling_style             0.000000
bedrooms                   0.000000
prop_visible_minorities    0.000000
dtype: float64

## 2. Prepare once, globally

Categoricals are cast a single time, before splitting — casting per fold would give LightGBM inconsistent category codes across folds.

In [4]:
df = prepare(df, cfg)
df.dtypes

house_price                 float64
dwelling_size               float64
property_age                float64
dwelling_style             category
bedrooms                    float64
bathrooms                   float64
prop_visible_minorities     float64
dtype: object

## 3. The training-time filter

The >$3M cap (and the $1k floor that protects MAPE from $0/$1 non-arm's-length
transfers) applies to **training rows only**. Validation folds stay unfiltered so
the score reflects the population the model actually sees at inference.

In [5]:
print("all rows:      ", len(df))
print("training rows: ", len(train_filter(df, cfg)))
print("validation max price stays:", df[TARGET].max())

all rows:       3000
training rows:  3000
validation max price stays: 1113172.542626671


## 4. Backtest splits

The municipality snapshot has no time axis, so this is seeded K-fold. With a date column, `Config(split="expanding", date_col="sale_date")` switches to an expanding window.

In [6]:
splits = make_splits(df, cfg)
[(len(tr), len(va)) for tr, va in splits]

[(2400, 600), (2400, 600), (2400, 600), (2400, 600), (2400, 600)]

## 5. Why LightGBM — the empirical comparison

Every candidate is scored on the identical folds with the identical filter and metric.

In [7]:
table = benchmark(df, cfg)
table

[bench] lightgbm       mape=0.0757 (+/-0.0028)  2.1s


[bench] xgboost        mape=0.0805 (+/-0.0028)  0.7s


[bench] hist_gbm       mape=0.0750 (+/-0.0035)  1.5s
[bench] ridge_imputed  mape=0.0865 (+/-0.0034)  0.0s


,model,cv_mape,cv_mape_std,fit_seconds
0,hist_gbm,0.074981,0.003479,1.505527
1,lightgbm,0.075705,0.002813,2.084186
2,xgboost,0.080460,0.002829,0.684108
3,ridge_imputed,0.086470,0.003366,0.019383


## 6. Tune (TPE, not grid search) and backtest the winner

In [8]:
cfg.n_trials = 12  # raise for a real run
study = tune(df, splits, cfg)
best = base_params(cfg) | study.best_params | {"subsample_freq": 1}
cv_mape, fold_scores = cv_score(best, df, splits, cfg)
print(f"cv mape = {cv_mape:.4f}")
pd.Series(fold_scores, name="fold_mape")

cv mape = 0.0713


0    0.067518
1    0.069239
2    0.070432
3    0.073364
4    0.075828
Name: fold_mape, dtype: float64

## 7. Fit final model and inspect drivers

In [9]:
model = fit_final(best, df, cfg)
pd.Series(model.feature_importances_, index=cfg.features).sort_values(ascending=False)

dwelling_size              14973
prop_visible_minorities    13441
property_age               13076
bedrooms                    4027
bathrooms                   3012
dwelling_style              1933
dtype: int32

### A note on `prop_visible_minorities`

It is in the feature set as specified, but using it in an automated valuation
model is a fair-housing / disparate-impact exposure. Drop it from `FEATURES`
in `config.py` (one line) to exclude it, then re-run this notebook to see the
MAPE cost of doing so.

## 8. Versioned artifacts

Each run lands in `<artifact_dir>/<package version>-<git sha>/`, so a model on disk is traceable to the code that produced it.

In [10]:
out = write_artifacts(model, cfg, study.best_params, cv_mape, fold_scores, splits)
sorted(p.name for p in out.iterdir())

[artifacts] written to /tmp/house_price_model/0.1.0-83f95a6


['model.joblib',
 'model_choice.json',
 'model_comparison.csv',
 'run.json',
 'splits.npz']